In [ ]:
#######################################################################################################################################################################
########################################### This notebook takes the cleaned data file and format it for Google Earth Engine ##########################################
#######################################################################################################################################################################

In [1]:
# Import the necessary packages
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import ee # Import the Python API package
import geemap # The geemap Python package is built upon the ipyleaflet and folium packages and implements several methods for interacting with Earth Engine data layer

try:
    ee.Initialize(project='amibea') # Initialize the API
except Exception as e:
    ee.Authenticate() # Authenticate to the Earth Engine servers
    ee.Initialize(project='amibea') # Initialize the API

C:\python\anaconda3\envs\Amibea\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Enter verification code:  4/1ATX87lPNDktOSMGg_JLit19Rb9e4bjoKj-joDqFb07NGCsrhuKNnRxGg3xI



Successfully saved authorization token.


In [3]:
# Import the raw data from a CSV file into a data frame
pathToFile = r'E:\Amibatec\Updated2026\TA_Clusters.csv' # Change path to your data
rawData = pd.read_csv(pathToFile,sep=";",encoding= 'unicode_escape')
Data = pd.DataFrame(rawData)
print(Data.shape)
print(Data.head(10))

(1300, 5)
    latitude   longitude  cluster1per  cluster2per  cluster3per
0  80.183543   52.853088     0.250000     0.875000     0.937500
1  79.866876   10.298928     0.235294     0.529412     1.235294
2  79.491876   13.032261     0.083333     1.000000     0.750000
3  79.452293   13.284344     0.555556     0.555556     0.333333
4  79.429376   13.280177     1.000000     0.000000     0.000000
5  79.146043  102.723913     0.285714     0.857143     0.714286
6  78.908543   12.076011     0.000000     0.666667     2.000000
7  78.902293   12.113511     0.000000     1.200000     1.200000
8  78.902293   12.105178     0.000000     0.857143     1.714286
9  78.900210   12.105178     0.000000     1.333333     1.000000


In [5]:
# Export the result to disc
Data.to_csv(r'E:\Amibatec\Updated2026\TA_Clusters_GEEformated.csv', index = False)

In [6]:
# Export an ee.FeatureCollection as an Earth Engine asset.
# create a tmp gdf

df = pd.read_csv(r'E:\Amibatec\Updated2026\TA_Clusters_GEEformated.csv', sep=',', engine='python')
gdf = gpd.GeoDataFrame(
    df, 
    crs='EPSG:4326', 
    geometry = gpd.points_from_xy(
        df['longitude'], 
        df['latitude']
    )
)
    
# convert it into geo-json 
json_df = json.loads(gdf.to_json())
    
# create a gee object with geemap
ee_object = geemap.geojson_to_ee(json_df)
            
#create and launch the task
task = ee.batch.Export.table.toAsset(
    collection= ee_object,
    description= 'exportToTableAsset',
    assetId= 'projects/amibea/assets/TA_Clusters_GEEformated_2026',
)
task.start()